# Product Image Processing Notebook

This notebook is designed to:
1. Load product image data from JSON files
2. Enrich image data with product information
3. Download images from URLs and organize them in a folder structure
4. Convert images to JPEG format for AI processing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Required Libraries

First, let's import all the libraries we'll need for this processing task.

In [ ]:
# Import necessary libraries
import requests
import json
import os
import re
from bs4 import BeautifulSoup
import time
from urllib.parse import urlparse
from typing import Dict, List
import shutil
from PIL import Image
import io
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

## Define File Paths and Create Folders

Set up our file paths and ensure all required folders exist.

In [ ]:
BASE_FOLDER = "drive/MyDrive/KLTN_DATA/"

# Input files
INPUT_FILE = BASE_FOLDER + "cleaned_grouped_products.json"
CATEGORIES_INPUT = BASE_FOLDER + "categories.json"

# Output files
OUTPUT_FOLDER = BASE_FOLDER + "output/"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

IMAGES_FOLDER = OUTPUT_FOLDER + "images/"
if not os.path.exists(IMAGES_FOLDER):
    os.makedirs(IMAGES_FOLDER)

PRODUCTS_FILE = OUTPUT_FOLDER + "products.json"
VARIANTS_FILE = OUTPUT_FOLDER + "product_variants.json"
IMAGES_FILE = OUTPUT_FOLDER + "product_images.json"
IMAGES_ENRICHED_FILE = OUTPUT_FOLDER + "product_images_injected.json"
AUTHORS_FILE = OUTPUT_FOLDER + "book_authors.json"
PRODUCT_OPTIONS_FILE = OUTPUT_FOLDER + "product_options.json"
PRODUCT_OPTION_VALUES_FILE = OUTPUT_FOLDER + "product_option_values.json"

print("Folders and file paths defined successfully.")

Folders and file paths defined successfully.


## Load Product and Image Data

In this section, we'll load the product data and product images data from JSON files.

In [ ]:
def load_json_file(file_path):
    """Load data from a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load product data
products = load_json_file(PRODUCTS_FILE)
product_images = load_json_file(IMAGES_FILE)

# Create a dictionary for fast product lookup
products_dict = {product['Id']: product for product in products}

print(f"Loaded {len(products)} products and {len(product_images)} product images.")

Loaded 2351 products and 25139 product images.


## Enrich Product Images with Product Information

Add product name and description to each image record.

In [ ]:
def enrich_product_images(images, products_dict):
    """Add product name and description to each image record."""
    enriched_images = []
    missing_products = set()

    for image in images:
        product_id = image['ProductId']
        product = products_dict.get(product_id)

        if product:
            enriched_image = image.copy()
            enriched_image['ProductName'] = product['Name']
            enriched_image['ProductDescription'] = product['Description']
            enriched_images.append(enriched_image)
        else:
            missing_products.add(product_id)

    if missing_products:
        print(f"Warning: {len(missing_products)} product IDs not found in products data.")
        print(f"Missing product IDs: {list(missing_products)[:10]}... (showing max 10)")

    return enriched_images

# Enrich product images with product data
enriched_images = enrich_product_images(product_images, products_dict)

# Save enriched images to a new file
with open(IMAGES_ENRICHED_FILE, 'w', encoding='utf-8') as f:
    json.dump(enriched_images, f, ensure_ascii=False, indent=4)

print(f"Enriched {len(enriched_images)} images with product info and saved to {IMAGES_ENRICHED_FILE}")

Missing product IDs: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]... (showing max 10)
Enriched 104 images with product info and saved to drive/MyDrive/KLTN_DATA/output/product_images_injected.json


## Download Images

Download images from URLs and organize them into folders by product ID.

In [ ]:
def get_file_extension(url):
    """Extract file extension from URL, defaulting to jpeg if not found."""
    parsed = urlparse(url)
    path = parsed.path
    extension = os.path.splitext(path)[1].lower()
    if extension and extension[1:] in ['jpg', 'jpeg', 'png', 'gif', 'bmp', 'webp']:
        return extension
    return '.jpeg'  # Default extension

def download_and_convert_image(image_data, index):
    """Download an image and convert it to JPEG if needed."""
    product_id = image_data['ProductId']
    image_id = image_data['Id']
    url = image_data['LargeImageUrl']

    # Create product folder if it doesn't exist
    product_folder = os.path.join(IMAGES_FOLDER, f"prod_{product_id}")
    if not os.path.exists(product_folder):
        os.makedirs(product_folder)

    # Format image number with leading zeros
    image_number = str(index + 1).zfill(3)
    image_path = os.path.join(product_folder, f"img_{product_id}_{image_number}.jpeg")

    # Skip if image already exists
    if os.path.exists(image_path):
        return f"Image already exists: {image_path}"

    try:
        # Download the image
        response = requests.get(url, stream=True, timeout=10)
        if response.status_code != 200:
            return f"Failed to download {url}: HTTP {response.status_code}"

        # Convert the image to JPEG
        img = Image.open(io.BytesIO(response.content))

        # Convert RGBA to RGB if needed (JPEG doesn't support alpha channel)
        if img.mode == 'RGBA':
            canvas = Image.new('RGB', img.size, (255, 255, 255))
            canvas.paste(img, mask=img.split()[3])
            img = canvas
        elif img.mode != 'RGB':
            img = img.convert('RGB')

        # Save as JPEG
        img.save(image_path, 'JPEG', quality=90)

        return f"Downloaded and converted: {image_path}"
    except Exception as e:
        return f"Error processing {url}: {str(e)}"

def download_all_images(images, max_workers=10):
    """Download all images using thread pool for parallel processing."""
    print(f"Downloading {len(images)} images with {max_workers} parallel workers...")

    # Group images by product ID to process them together
    images_by_product = {}
    for img in images:
        product_id = img['ProductId']
        if product_id not in images_by_product:
            images_by_product[product_id] = []
        images_by_product[product_id].append(img)

    # Flatten the list but keep the order by product
    ordered_images = []
    for product_id, product_images in images_by_product.items():
        ordered_images.extend(product_images)

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_image = {executor.submit(download_and_convert_image, img, idx): img
                          for idx, img in enumerate(ordered_images)}

        for future in tqdm(future_to_image, total=len(future_to_image)):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                results.append(f"Exception occurred: {str(e)}")

    return results

In [ ]:
# Download the images
results = download_all_images(enriched_images, max_workers=20)

# Count successes and failures
successes = sum(1 for r in results if r.startswith("Downloaded"))
already_exists = sum(1 for r in results if r.startswith("Image already exists"))
failures = len(results) - successes - already_exists

print(f"\nDownload summary:")
print(f"  - Successfully downloaded and converted: {successes}")
print(f"  - Already existing: {already_exists}")
print(f"  - Failed: {failures}")

# Print some failures if any
if failures > 0:
    print("\nSample failures:")
    for r in results:
        if not (r.startswith("Downloaded") or r.startswith("Image already exists")):
            print(f"  - {r}")
            if results.index(r) > 10:
                print("  - ...more failures")
                break

  0%|          | 0/104 [00:00<?, ?it/s]


Download summary:
  - Successfully downloaded and converted: 104
  - Already existing: 0
  - Failed: 0


## Validate Downloaded Images

Check if all images were downloaded successfully and generate statistics.

In [ ]:
def validate_images_folder():
    """Check the downloaded images and generate statistics."""
    product_folders = [f for f in os.listdir(IMAGES_FOLDER) if f.startswith('prod_')]

    total_products = len(product_folders)
    total_images = 0
    products_with_images = {}

    for folder in product_folders:
        product_path = os.path.join(IMAGES_FOLDER, folder)
        if os.path.isdir(product_path):
            product_id = folder.replace('prod_', '')
            images = [f for f in os.listdir(product_path) if f.endswith('.jpeg')]
            products_with_images[product_id] = len(images)
            total_images += len(images)

    # Calculate statistics
    avg_images_per_product = total_images / total_products if total_products > 0 else 0
    products_with_no_images = sum(1 for count in products_with_images.values() if count == 0)
    products_with_one_image = sum(1 for count in products_with_images.values() if count == 1)
    products_with_multiple_images = sum(1 for count in products_with_images.values() if count > 1)

    print(f"Image Download Statistics:")
    print(f"  - Total products with folders: {total_products}")
    print(f"  - Total downloaded images: {total_images}")
    print(f"  - Average images per product: {avg_images_per_product:.2f}")
    print(f"  - Products with no images: {products_with_no_images}")
    print(f"  - Products with exactly one image: {products_with_one_image}")
    print(f"  - Products with multiple images: {products_with_multiple_images}")

    # Return products with most images (top 10)
    top_products = sorted(products_with_images.items(), key=lambda x: x[1], reverse=True)[:10]
    print("\nProducts with most images:")
    for product_id, count in top_products:
        product_name = products_dict.get(int(product_id), {}).get('Name', 'Unknown')
        print(f"  - Product {product_id} ({product_name}): {count} images")

    return products_with_images

# Validate the images folder
products_with_images = validate_images_folder()

Image Download Statistics:
  - Total products with folders: 5
  - Total downloaded images: 104
  - Average images per product: 20.80
  - Products with no images: 0
  - Products with exactly one image: 0
  - Products with multiple images: 5

Products with most images:
  - Product 3 (Hướng Dẫn Bài Bản Ultimate Guide Series): 44 images
  - Product 2 (HBR Onpoint 2021): 25 images
  - Product 1 (Hiệu Ứng Chim Mồi): 22 images
  - Product 5 (Mực Lọ): 9 images
  - Product 4 (Mực Lông Dầu Horse): 4 images


## Summary and Next Steps

The notebook has completed the following tasks:
1. Loaded product and image data from JSON files
2. Enriched image data with product names and descriptions
3. Downloaded and converted all product images to JPEG format
4. Organized images into folders by product ID
5. Validated the downloaded images and generated statistics

These images are now ready for AI processing such as vector embedding for image search functionality.

In [ ]:
# Create a report of all processed images with their paths
def generate_image_report():
    report = []
    for product_id, image_count in products_with_images.items():
        if image_count > 0:
            product_name = products_dict.get(int(product_id), {}).get('Name', 'Unknown')
            product_folder = os.path.join(IMAGES_FOLDER, f"prod_{product_id}")
            images = [f for f in os.listdir(product_folder) if f.endswith('.jpeg')]

            for image in images:
                image_path = os.path.join(product_folder, image)
                report.append({
                    'product_id': product_id,
                    'product_name': product_name,
                    'image_path': image_path,
                    'image_filename': image
                })

    report_file = os.path.join(OUTPUT_FOLDER, 'image_report.json')
    with open(report_file, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=4)

    print(f"Image report generated with {len(report)} entries and saved to {report_file}")
    return report

# Generate the image report
image_report = generate_image_report()

Image report generated with 104 entries and saved to drive/MyDrive/KLTN_DATA/output/image_report.json


## Scrape Product Attributes

In this section, we'll scrape product attribute data from Fahasa product detail pages.
We'll create four files:
1. ProductTypeAttribute.json - Contains attribute definitions (e.g., Publisher, Year, Format)
2. ProductTypeAttributeValue.json - Contains unique attribute values
3. ProductTypeAttributeMapping.json - Maps attributes to product types
4. ProductTypeAttributeProductValue.json - Maps specific attribute values to products

In [ ]:
# Define output paths for attribute files
PRODUCT_TYPE_ATTRIBUTE_FILE = OUTPUT_FOLDER + "ProductTypeAttribute.json"
PRODUCT_TYPE_ATTRIBUTE_VALUE_FILE = OUTPUT_FOLDER + "ProductTypeAttributeValue.json"
PRODUCT_TYPE_ATTRIBUTE_MAPPING_FILE = OUTPUT_FOLDER + "ProductTypeAttributeMapping.json"
PRODUCT_TYPE_ATTRIBUTE_PRODUCT_VALUE_FILE = OUTPUT_FOLDER + "ProductTypeAttributeProductValue.json"

# Load the cleaned_grouped_products.json file to get product URLs
cleaned_products = load_json_file(INPUT_FILE)
categories = load_json_file(CATEGORIES_INPUT)

if not cleaned_products:
    print("Error: Could not load cleaned_grouped_products.json")
else:
    print(f"Loaded {len(cleaned_products)} grouped products")

# Create product name to ID and product ID to type ID mappings
product_name_to_id = {product['Name']: product['Id'] for product in products}
product_id_to_type = {product['Id']: product['ProductTypeId'] for product in products}

# Initialize attribute dictionaries and lists
attributes = []
attribute_values = []
attribute_mappings = []
product_attribute_values = []

# Track next available IDs
next_attribute_id = 1
next_attribute_value_id = 1
next_attribute_mapping_id = 1
next_product_attribute_value_id = 1

# Create dictionaries to track existing entries
attribute_name_to_id = {}
attribute_value_key_to_id = {}  # Key is "attribute_id:value"
type_attribute_key_to_id = {}   # Key is "type_id:attribute_id"

# Define helper functions that use global variables instead of nonlocal
def get_or_create_attribute(name):
    """Get existing attribute ID or create a new one"""
    global next_attribute_id, attributes, attribute_name_to_id

    if name in attribute_name_to_id:
        return attribute_name_to_id[name]

    attribute_id = next_attribute_id
    next_attribute_id += 1

    attributes.append({
        "Id": attribute_id,
        "Name": name
    })

    attribute_name_to_id[name] = attribute_id
    return attribute_id

def get_or_create_attribute_value(attribute_id, value):
    """Get existing attribute value ID or create a new one"""
    global next_attribute_value_id, attribute_values, attribute_value_key_to_id

    key = f"{attribute_id}:{value}"
    if key in attribute_value_key_to_id:
        return attribute_value_key_to_id[key]

    value_id = next_attribute_value_id
    next_attribute_value_id += 1

    attribute_values.append({
        "Id": value_id,
        "ProductTypeAttributeId": attribute_id,
        "Value": value
    })

    attribute_value_key_to_id[key] = value_id
    return value_id

def get_or_create_type_attribute_mapping(type_id, attribute_id):
    """Get existing type-attribute mapping ID or create a new one"""
    global next_attribute_mapping_id, attribute_mappings, type_attribute_key_to_id

    key = f"{type_id}:{attribute_id}"
    if key in type_attribute_key_to_id:
        return type_attribute_key_to_id[key]

    mapping_id = next_attribute_mapping_id
    next_attribute_mapping_id += 1

    attribute_mappings.append({
        "Id": mapping_id,
        "ProductTypeId": type_id,
        "ProductTypeAttributeId": attribute_id
    })

    type_attribute_key_to_id[key] = mapping_id
    return mapping_id

def create_product_attribute_value(product_id, attribute_value_id):
    """Create a product-attribute-value mapping"""
    global next_product_attribute_value_id, product_attribute_values

    product_attribute_values.append({
        "Id": next_product_attribute_value_id,
        "ProductId": product_id,
        "AttributeValueId": attribute_value_id
    })

    next_product_attribute_value_id += 1

# Function to scrape attributes from a product page
def scrape_product_attributes(product_url, product_name):
    if product_name not in product_name_to_id:
        print(f"Warning: Product '{product_name}' not found in products.json")
        return

    product_id = product_name_to_id[product_name]
    product_type_id = product_id_to_type[product_id]

    print(f"Scraping attributes for: {product_name} (ID: {product_id}, Type: {product_type_id})")

    try:
        # Add delay to avoid overloading the server
        time.sleep(1)

        # Fetch the product page
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(product_url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"Error: Failed to fetch {product_url} - Status code: {response.status_code}")
            return

        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the product attributes table
        attribute_table = soup.select_one('.product_view_tab_content_additional .table-additional')

        if not attribute_table:
            print(f"Warning: No attribute table found for {product_name}")
            return

        # Extract attribute name-value pairs
        rows = attribute_table.select('tr')
        for row in rows:
            label_elem = row.select_one('.table-label')
            value_elem = row.select_one('.attribute_link_container')

            if label_elem and value_elem:
                attribute_name = label_elem.get_text(strip=True)
                attribute_value = value_elem.get_text(strip=True)

                # Skip empty values
                if not attribute_value:
                    continue

                # Get or create attribute
                attribute_id = get_or_create_attribute(attribute_name)

                # Get or create attribute value
                attribute_value_id = get_or_create_attribute_value(attribute_id, attribute_value)

                # Get or create type-attribute mapping
                get_or_create_type_attribute_mapping(product_type_id, attribute_id)

                # Create product-attribute-value mapping
                create_product_attribute_value(product_id, attribute_value_id)

        print(f"Successfully scraped attributes for {product_name}")
    except Exception as e:
        print(f"Error scraping {product_url}: {str(e)}")

# Process each product
processed_count = 0
for product_group in tqdm(cleaned_products[:30], desc="Scraping product attributes"):  # Limiting to first 30 for demonstration
    base_product_name = product_group.get('base_product_name')
    variants = product_group.get('variants', [])

    if not variants:
        continue

    # Get the URL of the first variant
    first_variant = variants[0]
    product_url = first_variant.get('product_url')

    if not product_url:
        continue

    # Scrape attributes for this product
    scrape_product_attributes(product_url, base_product_name)
    processed_count += 1

print(f"\nProcessed {processed_count} products")
print(f"Found {len(attributes)} unique attributes")
print(f"Found {len(attribute_values)} unique attribute values")
print(f"Created {len(attribute_mappings)} attribute-type mappings")
print(f"Created {len(product_attribute_values)} product-attribute-value assignments")

# Save the results to JSON files
def save_to_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Saved {len(data)} items to {file_path}")

save_to_json(attributes, PRODUCT_TYPE_ATTRIBUTE_FILE)
save_to_json(attribute_values, PRODUCT_TYPE_ATTRIBUTE_VALUE_FILE)
save_to_json(attribute_mappings, PRODUCT_TYPE_ATTRIBUTE_MAPPING_FILE)
save_to_json(product_attribute_values, PRODUCT_TYPE_ATTRIBUTE_PRODUCT_VALUE_FILE)

Loaded 2352 grouped products


Scraping product attributes:   0%|          | 0/30 [00:00<?, ?it/s]

Scraping attributes for: Hiệu Ứng Chim Mồi (ID: 1, Type: 9)
Successfully scraped attributes for Hiệu Ứng Chim Mồi
Scraping attributes for: HBR Onpoint 2021 (ID: 2, Type: 9)
Successfully scraped attributes for HBR Onpoint 2021
Scraping attributes for: Hướng Dẫn Bài Bản Ultimate Guide Series (ID: 3, Type: 9)
Successfully scraped attributes for Hướng Dẫn Bài Bản Ultimate Guide Series
Scraping attributes for: Mực Lông Dầu Horse (ID: 4, Type: 63)
Successfully scraped attributes for Mực Lông Dầu Horse
Scraping attributes for: Mực Lọ (ID: 5, Type: 63)
Successfully scraped attributes for Mực Lọ
Scraping attributes for: Mực Máy Bấm (ID: 6, Type: 63)
Successfully scraped attributes for Mực Máy Bấm
Scraping attributes for: Mực Viết Bảng Artline ESK-50 (ID: 7, Type: 63)
Successfully scraped attributes for Mực Viết Bảng Artline ESK-50
Scraping attributes for: Pentel Phấn Dầu PHN-50 (ID: 8, Type: 63)
Successfully scraped attributes for Pentel Phấn Dầu PHN-50
Scraping attributes for: Phấn Vặn Đa Năng

## Scrape Product Variants

This section processes variants data to create:
1. ProductVariants.json - Contains variant details like price and stock
2. ProductOptions.json - Contains option types (e.g., "Tập", "Bản")
3. ProductOptionValues.json - Contains option values (e.g., "2", "Tái Bản 2023")
4. ProductVariantOptionValues.json - Maps variants to their option values

In [ ]:
# Define output paths for variant-related files
PRODUCT_VARIANTS_FILE = OUTPUT_FOLDER + "product_variants.json"
PRODUCT_OPTIONS_FILE = OUTPUT_FOLDER + "product_options.json"
PRODUCT_OPTION_VALUES_FILE = OUTPUT_FOLDER + "product_option_values.json"
PRODUCT_VARIANT_OPTION_VALUES_FILE = OUTPUT_FOLDER + "product_variant_option_values.json"

# If product options and values don't exist yet, initialize them
product_options = []
product_option_values = []
product_variants = []
product_variant_option_values = []

# Track next available IDs
next_variant_id = 1
next_option_id = 1
next_option_value_id = 1
next_variant_option_value_id = 1

# Create dictionaries to track existing entries
option_key_to_id = {}  # Key is "product_id:name"
option_value_key_to_id = {}  # Key is "option_id:value"

# Helper functions for product variants
def get_or_create_option(product_id, option_name):
    """Get existing option ID or create a new one"""
    global next_option_id, product_options, option_key_to_id
    
    key = f"{product_id}:{option_name}"
    if key in option_key_to_id:
        return option_key_to_id[key]
    
    option_id = next_option_id
    next_option_id += 1
    
    product_options.append({
        "Id": option_id,
        "ProductId": product_id,
        "Name": option_name,
        "IsOptionWithImage": True  # Default to True as these are typically visual options
    })
    
    option_key_to_id[key] = option_id
    return option_id

def get_or_create_option_value(option_id, value):
    """Get existing option value ID or create a new one"""
    global next_option_value_id, product_option_values, option_value_key_to_id
    
    key = f"{option_id}:{value}"
    if key in option_value_key_to_id:
        return option_value_key_to_id[key]
    
    value_id = next_option_value_id
    next_option_value_id += 1
    
    product_option_values.append({
        "Id": value_id,
        "OptionId": option_id,
        "Value": value,
        "ThumbnailImageUrl": "",
        "LargeImageUrl": ""
    })
    
    option_value_key_to_id[key] = value_id
    return value_id

def create_variant_option_value(variant_id, option_id, option_value_id):
    """Create a product variant option value mapping"""
    global next_variant_option_value_id, product_variant_option_values
    
    product_variant_option_values.append({
        "Id": next_variant_option_value_id,
        "ProductVariantId": variant_id,
        "OptionId": option_id,
        "OptionValueId": option_value_id
    })
    
    next_variant_option_value_id += 1

def extract_price_from_text(price_text):
    """Extract numeric price from text like '56.000 đ'"""
    if not price_text:
        return 0
    
    # Remove currency symbol and non-numeric characters except for decimal point
    numeric_text = re.sub(r'[^\d.]', '', price_text.replace(',', '.').replace('đ', ''))
    
    try:
        # Convert to float and then to integer (remove decimal part)
        price = int(float(numeric_text))
        return price
    except ValueError:
        return 0

def generate_random_barcode():
    """Generate a random 13-character barcode"""
    return ''.join(str(random.randint(0, 9)) for _ in range(13))

def scrape_variant_data(product_url, product_id, variant_attributes):
    """Scrape variant data from product URL"""
    global next_variant_id
    
    variant_id = next_variant_id
    next_variant_id += 1
    
    print(f"Scraping variant data for product ID {product_id} from {product_url}")
    
    # Default values
    unit_price = 0
    recommended_retail_price = 0
    weight = 0
    quantity = random.randint(10, 50)  # Random stock between 10-50
    barcode = generate_random_barcode()
    
    try:
        # Add delay to avoid overloading the server
        time.sleep(1)
        
        # Fetch the product page
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(product_url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Error: Failed to fetch {product_url} - Status code: {response.status_code}")
        else:
            # Parse the HTML content
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract special price (unit price)
            special_price_element = soup.select_one('div[id="catalog-product-details-price"] p.special-price span.price:not(.label)')
            if special_price_element:
                unit_price = extract_price_from_text(special_price_element.get_text(strip=True))
            
            # Extract old price (recommended retail price)
            old_price_element = soup.select_one('div[id="catalog-product-details-price"] p.old-price span.price:not(.label)')
            if old_price_element:
                recommended_retail_price = extract_price_from_text(old_price_element.get_text(strip=True))
            else:
                # If no old price, use the unit price as recommended price
                recommended_retail_price = unit_price
            
            # Extract weight
            weight_row = soup.select_one('tr:has(th:contains("Trọng lượng"))')
            if weight_row:
                weight_div = weight_row.select_one('td div')
                if weight_div:
                    weight_text = weight_div.get_text(strip=True)
                    weight_match = re.search(r'\d+', weight_text)
                    if weight_match:
                        weight = int(weight_match.group())
    
    except Exception as e:
        print(f"Error scraping {product_url}: {str(e)}")
    
    # Create variant
    variant = {
        "Id": variant_id,
        "ProductId": product_id,
        "UnitPrice": unit_price,
        "RecommendedRetailPrice": recommended_retail_price,
        "ValidFrom": datetime.datetime.now().isoformat(),
        "Quantity": quantity,
        "Barcode": barcode,
        "Weight": weight,
        "Comment": "Good",
        "Tags": ""
    }
    
    product_variants.append(variant)
    
    # Process variant attributes to create options and option values
    for option_name, option_value in variant_attributes.items():
        option_id = get_or_create_option(product_id, option_name)
        option_value_id = get_or_create_option_value(option_id, option_value)
        create_variant_option_value(variant_id, option_id, option_value_id)
    
    return variant_id

# Ensure we have the required libraries for this section
import random
import datetime

# Load cleaned grouped products if not loaded already
if not cleaned_products:
    cleaned_products = load_json_file(INPUT_FILE)
    if not cleaned_products:
        print("Error: Could not load cleaned_grouped_products.json")

# Process each product group to extract variants
processed_variant_count = 0
for product_group in tqdm(cleaned_products[:4], desc="Processing product variants"):
    base_product_name = product_group.get('base_product_name')
    variants_data = product_group.get('variants', [])
    
    # Skip if no variants or no base product name
    if not variants_data or not base_product_name:
        continue
    
    # Find product ID using the base product name
    product_id = None
    for prod in products:
        if prod['Name'] == base_product_name:
            product_id = prod['Id']
            break
    
    if not product_id:
        print(f"Warning: Product '{base_product_name}' not found in products.json")
        continue
    
    # Process each variant
    for variant_data in variants_data:
        variant_attributes = variant_data.get('variant_attributes', {})
        product_url = variant_data.get('product_url')
        
        if not product_url:
            continue
        
        # Scrape variant data
        variant_id = scrape_variant_data(product_url, product_id, variant_attributes)
        processed_variant_count += 1

print(f"\nProcessed {processed_variant_count} product variants")
print(f"Created {len(product_options)} product options")
print(f"Created {len(product_option_values)} product option values")
print(f"Created {len(product_variant_option_values)} product variant option value mappings")

# Save the results to JSON files
save_to_json(product_variants, PRODUCT_VARIANTS_FILE)
save_to_json(product_options, PRODUCT_OPTIONS_FILE)
save_to_json(product_option_values, PRODUCT_OPTION_VALUES_FILE)
save_to_json(product_variant_option_values, PRODUCT_VARIANT_OPTION_VALUES_FILE)